In [1]:
import os
import glob
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

Kullanılan Cihaz: cpu


In [3]:
print("ResNet-18 Modeli yükleniyor...")
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights).to(device)
model.eval() # Test moduna al

ResNet-18 Modeli yükleniyor...


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
# 3. Görüntü Dönüşümleri (ResNet-18 resimleri 224x224 ve normalize bekler)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

In [5]:
# 4. Altın Standart (Temiz Resimler) Tahminlerini Al
print("\nAdım 1: Orijinal (Temiz) resimlerin referans tahminleri alınıyor...")
baseline_preds = {}
original_folder = 'Tiny_Orijinal'

# Hem .png hem .JPEG/jpg dosyalarını bulalım
orig_images = []
for ext in ['*.png', '*.JPEG', '*.jpg']:
    orig_images.extend(glob.glob(os.path.join(original_folder, ext)))
if len(orig_images) == 0:
    print(f"Hata: {original_folder} klasöründe resim bulunamadı!")
with torch.no_grad():
    for img_path in tqdm(orig_images, desc="Baseline Çıkarımı"):
        img_name_no_ext = os.path.splitext(os.path.basename(img_path))[0]
        
        img = Image.open(img_path).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(device)
        
        output = model(tensor)
        _, pred_class = output.max(1)
        baseline_preds[img_name_no_ext] = pred_class.item()

total_images = len(baseline_preds)
print(f"Toplam {total_images} resim referans olarak kaydedildi.")


Adım 1: Orijinal (Temiz) resimlerin referans tahminleri alınıyor...


Baseline Çıkarımı: 100%|██████████| 1000/1000 [01:29<00:00, 11.24it/s]

Toplam 1000 resim referans olarak kaydedildi.


In [6]:
noise_levels = [10, 30, 50, 70, 90]
methods = {
        # 'Noisy': 'Tiny_Noisy_{}',
        # 'SMF': 'Tiny_Cleaned_SMF_{}',
        # 'AMF': 'Tiny_Cleaned_AMF_{}',
        # 'MDBUTMF': 'Tiny_Cleaned_MDBUTMF_{}',
        # 'DnCNN': 'Tiny_Cleaned_DnCNN_{}',
        # 'EMPR': 'Tiny_Cleaned_EMPR_{}',
        'SeConvUNet': 'ai_based_models/outputs/Tiny_Cleaned_SeConvUNet_{}'
    }

results = {method: [] for method in methods.keys()}

print("\nAdım 2: Temizleme algoritmaları test ediliyor...")

with torch.no_grad():
    for level in noise_levels:
        print(f"\n--- Gürültü Seviyesi: %{level} ---")
        for method_name, folder_template in methods.items():
            folder_path = folder_template.format(level)
            
            if not os.path.exists(folder_path):
                results[method_name].append("N/A")
                continue
            
            correct_count = 0
            processed_count = 0
            
            test_images = glob.glob(os.path.join(folder_path, '*.png'))
            
            for img_path in test_images:
                img_name_no_ext = os.path.splitext(os.path.basename(img_path))[0]
                
                # Sadece baseline'da olan resimleri test et
                if img_name_no_ext in baseline_preds:
                    img = Image.open(img_path).convert('RGB')
                    tensor = transform(img).unsqueeze(0).to(device)
                    
                    output = model(tensor)
                    _, pred_class = output.max(1)
                    
                    if pred_class.item() == baseline_preds[img_name_no_ext]:
                        correct_count += 1
                    processed_count += 1
            
            if processed_count > 0:
                accuracy = (correct_count / processed_count) * 100
                results[method_name].append(accuracy)
                print(f"{method_name:15s}: {accuracy:.2f}% doğruluk ({correct_count}/{processed_count})")
            else:
                results[method_name].append("N/A")


Adım 2: Temizleme algoritmaları test ediliyor...

--- Gürültü Seviyesi: %10 ---
SeConvUNet     : 2.60% doğruluk (26/1000)

--- Gürültü Seviyesi: %30 ---
SeConvUNet     : 1.20% doğruluk (12/1000)

--- Gürültü Seviyesi: %50 ---
SeConvUNet     : 0.60% doğruluk (6/1000)

--- Gürültü Seviyesi: %70 ---
SeConvUNet     : 0.10% doğruluk (1/1000)

--- Gürültü Seviyesi: %90 ---
SeConvUNet     : 0.00% doğruluk (0/1000)


In [7]:
print("\n" + "="*70)
print("RESNET-18 SINIFLANDIRMA DOĞRULUK (ACCURACY) TABLOSU")
print("="*70)
header = f"{'Yöntem':<15} | {'%10':<8} | {'%30':<8} | {'%50':<8} | {'%70':<8} | {'%90':<8}"
print(header)
print("-" * 70)
    
for method_name, accuracies in results.items():
    row_str = f"{method_name:<15} | "
    for acc in accuracies:
        if isinstance(acc, str):
            row_str += f"{acc:<8} | "
        else:
            row_str += f"{acc:>6.2f}% | "
    print(row_str)
print("="*70)
print("* Not: Doğruluk, gürültülü/temizlenmiş görüntünün, orijinal temiz")
print("  görüntüyle aynı ResNet-18 sınıfını üretme oranını temsil eder.")


RESNET-18 SINIFLANDIRMA DOĞRULUK (ACCURACY) TABLOSU
Yöntem          | %10      | %30      | %50      | %70      | %90     
----------------------------------------------------------------------
SeConvUNet      |   2.60% |   1.20% |   0.60% |   0.10% |   0.00% | 
* Not: Doğruluk, gürültülü/temizlenmiş görüntünün, orijinal temiz
  görüntüyle aynı ResNet-18 sınıfını üretme oranını temsil eder.
